<div class="alert alert-block alert-danger">
Before you start running this notebook, create a Hail-enabled Dataproc cluster with a <b>high compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>JupyterLab Spark cluster</b> and, under <b>Software to install</b>, choose <b>Hail</b>.</li>
    <li>Configure a high-compute cluster (multiple workers). Cluster creation can take about 10 minutes.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a> and <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/cloud_app_types/dataproc/">Use Dataproc (Spark) and Hail</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook inspects GLnexus joint-call Hail MatrixTables in the Terra workspace. It reports sample and variant counts, applies QC, LD-prunes, runs PCA and relatedness (`pc_relate`), and includes Hail 1KG tutorial cells (including a toy GWAS) as a reference.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

Where a cell reads or writes data files, the path is written relative to `AOU_LR_DATA`, an environment variable that this notebook reads but never sets. Before running, set it to the location of the notebook's data (in the original workspace, the workspace bucket). For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
# standard imports
import os
import sys
import pandas as pd
import re

import math
import datetime
from dateutil import parser
from time import sleep
from typing import List
import pprint

from google.cloud import storage
my_storage_client = storage.Client()

from tqdm.notebook import tqdm

In [ ]:
import firecloud.api as fapi
import hail as hl

In [ ]:
# Load a Terra data table into a pandas DataFrame, with the entity ID in an entity:<type>_id column.
def load_table(namespace, workspace, root_entity_type):
    ent = fapi.get_entities(namespace, workspace, root_entity_type).json()
    tbl = pd.DataFrame(list(map(lambda e: e['attributes'], ent)))
    tbl[f"entity:{root_entity_type}_id"] = list(map(lambda f: f['name'], ent))

    return tbl

In [ ]:
current_account = fapi.whoami()

In [ ]:
# Workspace details Terra sets as environment variables.
bucket = os.environ['WORKSPACE_BUCKET']       # the workspace bucket
workspace = os.environ['WORKSPACE_NAME']      # workspace name
namespace = os.environ['WORKSPACE_NAMESPACE'] # workspace namespace

In [ ]:
# The GLnexus joint-calling sample sets. Each row's joint_mt column is the path of its Hail MatrixTable.
tbl = load_table(namespace, workspace, 'ha-sample-grch38_set')

In [ ]:
tbl

In [ ]:
# MatrixTables for the 704-sample (alpha1) and 1,040-sample (2023Q1) joint calls, and their sizes.
mt_704_path = list(tbl[tbl['entity:ha-sample-grch38_set_id'] == 'cohort_for_GLNexus_alpha1_704']['joint_mt'])[0]
mt_704_path

In [ ]:
mt_1040_path = list(tbl[tbl['entity:ha-sample-grch38_set_id'] == 'cohort_for_GLNexus_2023Q1_1040']['joint_mt'])[0]
mt_1040_path

In [ ]:
!gsutil du -hcs {mt_704_path}
!gsutil du -hcs {mt_1040_path}

In [ ]:
#hl.stop()

In [ ]:
# Start Hail, using GRCh38 coordinates.
hl.init(default_reference='GRCh38')

In [ ]:
from hail.plot import show
from pprint import pprint
hl.plot.output_notebook()

In [ ]:
# Sample and variant counts of each MatrixTable.
joint_mt_704 = hl.read_matrix_table(mt_704_path)

In [ ]:
print('Samples: %d  Variants: %d' % (joint_mt_704.count_cols(), joint_mt_704.count_rows()))

In [ ]:
joint_mt_1040 = hl.read_matrix_table(mt_1040_path)

In [ ]:
print('Samples: %d  Variants: %d' % (joint_mt_1040.count_cols(), joint_mt_1040.count_rows()))

In [ ]:
# joint_mt is not assigned in this copy. The cells below expect the 1,040-sample MatrixTable
# (joint_mt = joint_mt_1040); their outputs are written under cohort_for_GLNexus_2023Q1_1040.
hl.summarize_variants(joint_mt)

In [ ]:
# run sample QC and save into matrix table
joint_mt = hl.sample_qc(joint_mt)

In [ ]:
# run variant QC and save into matrix table
joint_mt = hl.variant_qc(joint_mt)

In [ ]:
# Autosomes only.
test_intervals = [
    'chr1',  'chr2',  'chr3',  'chr4',  'chr5',  'chr6',
    'chr7',  'chr8',  'chr9',  'chr10', 'chr11', 'chr12',
    'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18',
    'chr19', 'chr20', 'chr21', 'chr22'
] 
joint_mt = hl.filter_intervals(joint_mt, [hl.parse_locus_interval(x,) for x in test_intervals])

In [ ]:
# filter for variants that are >95% call rate, >1% frequency, and remove sites far from HWE
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.call_rate > 0.95)
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.AF[1] > 0.01)
joint_mt = joint_mt.filter_rows(joint_mt.variant_qc.p_value_hwe > 0.005)
joint_mt = joint_mt.filter_rows(hl.len(joint_mt.alleles) == 2)

In [ ]:
# LD-prune to roughly independent sites (r2 < 0.1 within 500 kb) for PCA and relatedness.
pruned_variant_table = hl.ld_prune(joint_mt.GT, r2=0.1, bp_window_size=500000)
joint_mt = joint_mt.filter_rows(hl.is_defined(pruned_variant_table[joint_mt.row_key]))

In [ ]:
print('Samples: %d  Variants: %d' % (joint_mt.count_cols(), joint_mt.count_rows()))

In [ ]:
hl.summarize_variants(joint_mt)

In [ ]:
import os

# Location of this notebook's data (a gs:// location). Set AOU_LR_DATA before running; see GETTING_STARTED.md.
AOU_LR_DATA = os.environ["AOU_LR_DATA"]

In [ ]:
# Save the pruned site table and MatrixTable.
pruned_variant_table.write(f"{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1040/pruned_tbl.ht")

In [ ]:
joint_mt.write(f"{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1040/pruned.mt", overwrite=True)

In [ ]:
!gsutil ls -lh {AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1040/
    

In [ ]:
pruned_mt = hl.read_matrix_table(f'{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1040/pruned.mt')

In [ ]:
hl.summarize_variants(pruned_mt)

In [ ]:
#rel = hl.pc_relate(pruned_mt.GT, 0.01, k=10, min_kinship=0.1)

In [ ]:
#pairs = rel.filter(rel['kin'] > 0.125)

In [ ]:
#related_samples_to_remove = hl.maximal_independent_set(pairs.i, pairs.j, keep=False)
#result = pruned_mt.filter_cols(hl.is_defined(related_samples_to_remove[mt.col_key]), keep=False)

In [ ]:
# PCA on the pruned sites, then kinship with PC-Relate using the top PCs as covariates. Pairs with
# kinship > 0.1 are reported, and a maximal independent set picks the samples to remove so that no
# related pair remains.
eigenvalues, pcs, _ = hl.hwe_normalized_pca(pruned_mt.GT, k=10, compute_loadings=False)

In [ ]:
rel = hl.pc_relate(pruned_mt.GT,
                   0.01,
                   scores_expr=pcs[pruned_mt.col_key].scores,
                   min_kinship=0.1)

In [ ]:
related_samples_to_remove = hl.maximal_independent_set(rel.i, rel.j, keep=False)

In [ ]:
related_samples_to_remove.count()

In [ ]:
# Save the list of related samples to remove.
related_samples_to_remove.write(f'{AOU_LR_DATA}/outputs/GRCh38/JointCallGVCFs/cohort_for_GLNexus_2023Q1_1040/related_samples_to_remove.ht')

In [ ]:
rel.show()

In [ ]:
pprint(eigenvalues)

In [ ]:
pcs.show(5, width=100)

In [ ]:
# Plot PC1 vs PC2.
pruned_mt = pruned_mt.annotate_cols(scores = pcs[pruned_mt.s].scores)

In [ ]:
p = hl.plot.scatter(pruned_mt.scores[0],
                    pruned_mt.scores[1],
                    title='PCA', xlabel='PC1', ylabel='PC2')
show(p)

In [ ]:
# Load the gnomAD HGDP + 1000 Genomes dense subset (a reference panel), and inspect it.
mt = hl.experimental.load_dataset(name='gnomad_hgdp_1kg_subset_dense',
                                  version='3.1.2',
                                  reference_genome='GRCh38',
                                  region='us',
                                  cloud='gcp')

In [ ]:
mt.rows().select().show(5)

In [ ]:
mt.row_key.show(5)

In [ ]:
mt.cols().count()

In [ ]:
# The remaining cells follow Hail's GWAS tutorial on its small 1000 Genomes example dataset (with a
# simulated caffeine-consumption phenotype). They are not part of the Phase 1 analysis.
hl.utils.get_1kg('data/')

In [ ]:
hl.import_vcf('data/1kg.vcf.bgz').write('data/1kg.mt', overwrite=True)

In [ ]:
mt = hl.read_matrix_table('data/1kg.mt')

In [ ]:
mt.rows().select().show(5)

In [ ]:
mt.row_key.show(5)

In [ ]:
mt.s.show(5)

In [ ]:
mt.entry.take(5)

In [ ]:
table = (hl.import_table('data/1kg_annotations.txt', impute=True)
         .key_by('Sample'))

In [ ]:
table.describe()

In [ ]:
table.show(width=100)

In [ ]:
print(mt.col.dtype)

In [ ]:
mt = mt.annotate_cols(pheno = table[mt.s])

In [ ]:
mt.col.describe()

In [ ]:
pprint(table.aggregate(hl.agg.counter(table.SuperPopulation)))

In [ ]:
pprint(table.aggregate(hl.agg.stats(table.CaffeineConsumption)))

In [ ]:
table.count()

In [ ]:
mt.count_cols()

In [ ]:
mt.aggregate_cols(hl.agg.counter(mt.pheno.SuperPopulation))

In [ ]:
pprint(mt.aggregate_cols(hl.agg.stats(mt.pheno.CaffeineConsumption)))

In [ ]:
snp_counts = mt.aggregate_rows(hl.agg.counter(hl.Struct(ref=mt.alleles[0], alt=mt.alleles[1])))
pprint(snp_counts)

In [ ]:
from collections import Counter
counts = Counter(snp_counts)
counts.most_common()

In [ ]:
p = hl.plot.histogram(mt.DP, range=(0,30), bins=30, title='DP Histogram', legend='DP')
show(p)

In [ ]:
mt.col.describe()

In [ ]:
mt = hl.sample_qc(mt)

In [ ]:
mt.col.describe()

In [ ]:
p = hl.plot.histogram(mt.sample_qc.call_rate, range=(.88,1), legend='Call Rate')
show(p)

In [ ]:
p = hl.plot.histogram(mt.sample_qc.gq_stats.mean, range=(10,70), legend='Mean Sample GQ')
show(p)

In [ ]:
p = hl.plot.scatter(mt.sample_qc.dp_stats.mean, mt.sample_qc.call_rate, xlabel='Mean DP', ylabel='Call Rate')
show(p)

In [ ]:
mt = mt.filter_cols((mt.sample_qc.dp_stats.mean >= 4) & (mt.sample_qc.call_rate >= 0.97))
print('After filter, %d/284 samples remain.' % mt.count_cols())

In [ ]:
ab = mt.AD[1] / hl.sum(mt.AD)

filter_condition_ab = ((mt.GT.is_hom_ref() & (ab <= 0.1)) |
                        (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75)) |
                        (mt.GT.is_hom_var() & (ab >= 0.9)))

fraction_filtered = mt.aggregate_entries(hl.agg.fraction(~filter_condition_ab))
print(f'Filtering {fraction_filtered * 100:.2f}% entries out of downstream analysis.')
mt = mt.filter_entries(filter_condition_ab)

In [ ]:
mt = hl.variant_qc(mt)

In [ ]:
mt.row.describe()

In [ ]:
mt = mt.filter_rows(mt.variant_qc.AF[1] > 0.01)

In [ ]:
mt = mt.filter_rows(mt.variant_qc.p_value_hwe > 1e-6)

In [ ]:
print('Samples: %d  Variants: %d' % (mt.count_cols(), mt.count_rows()))

In [ ]:
gwas = hl.linear_regression_rows(y=mt.pheno.CaffeineConsumption,
                                 x=mt.GT.n_alt_alleles(),
                                 covariates=[1.0])
gwas.row.describe()

In [ ]:
p = hl.plot.manhattan(gwas.p_value)
show(p)